# Validação do Forecast de Custos — jan a jun/2026

Compara o **custo unitário previsto** (Custo Média Móvel) com o **custo unitário real**, por Centro + Código + Mês, e mede se o script agrega valor frente à alternativa trivial de repetir o custo do mês anterior.

**Como ler os indicadores**

| Indicador | Responde |
|---|---|
| Acurácia (1−WMAPE) | Quanto do valor movimentado foi previsto corretamente |
| MAE (R$/un) | Distância média entre preço previsto e real |
| Viés | O forecast superestima ou subestima? |
| Hit Rate | % de previsões dentro da tolerância da classe |
| Ganho vs baseline | Quanto de desvio o script evita frente a não fazer nada |

**Ordem de execução:** rode todas as células de cima para baixo. Várias etapas sobrescrevem colunas de `base` — executar fora de ordem produz resultados inconsistentes.

## 1. Configuração

In [ ]:
import unicodedata
import numpy as np
import pandas as pd

# ---- Colunas de trabalho ----
COL_CENTRO   = "Centro"
COL_MATERIAL = "Codigo"
COL_MES      = "Mes"
COL_PREV     = "Custo_Previsto"    # custo unitário previsto pelo script
COL_REAL     = "Custo_Real"        # custo unitário real
COL_PROD     = "Producao"          # produção REALIZADA (nunca a prevista)
COL_BASE     = "Baseline_M1"       # custo real do mês anterior (calculado na seção 3)

ANO = 2026

# ---- Tolerância ----
MODO_TOLERANCIA = "percentual"     # "percentual" | "absoluto" | "combinado"
REGRA_COMBINADO = "ou"             # usada só no modo combinado
TOLERANCIAS     = {"A": 0.05, "B": 0.10, "C": 0.20}   # erro % sobre o preço
TOLERANCIAS_ABS = {"A": 0.10, "B": 0.10, "C": 0.10}   # erro em R$/un

# ---- Curva ABC (participação acumulada no valor movimentado) ----
CORTE_A = 0.80
CORTE_B = 0.95

TOP_N = 15

MESES_NUM = {"janeiro": 1, "fevereiro": 2, "marco": 3, "abril": 4,
             "maio": 5, "junho": 6, "julho": 7, "agosto": 8,
             "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12}

MAPA_FABRICAS = {}   # <-- preencha com o seu de-para de Fábrica -> Centro

pd.set_option("display.max_columns", None)

## 2. Carga e merge

Dois cuidados que causaram erro antes e estão corrigidos aqui:

- **Renomear por dicionário, nunca por posição.** `usecols` devolve as colunas na ordem do arquivo, não na ordem que você listou — atribuir `df.columns = [...]` embaralha os dados silenciosamente.
- **Deduplicar os dois lados** antes do merge. O forecast vem aberto por componente; se a base real também tiver mais de uma linha por chave, o merge vira many-to-many e multiplica as linhas.

In [ ]:
# ---------- Base real (BI) ----------
df_bi = pd.read_excel("dados_reais.xlsx")
df_bi = df_bi.rename(columns={
    "Fábrica": COL_CENTRO,
    "Código": COL_MATERIAL,
    "Produto": "Produto",
    "Custo Estoque (R$/T)": COL_REAL,
    "Produção (T)": COL_PROD,
    "Mes": COL_MES,
})[[COL_CENTRO, COL_MATERIAL, "Produto", COL_REAL, COL_PROD, COL_MES]]

df_bi[COL_REAL] = df_bi[COL_REAL] / 1000    # R$/T -> R$/kg
df_bi[COL_PROD] = df_bi[COL_PROD] * 1000    # T    -> kg
if MAPA_FABRICAS:
    df_bi[COL_CENTRO] = df_bi[COL_CENTRO].map(MAPA_FABRICAS)

# ---------- Base do forecast ----------
df_fc = pd.read_excel("forecast.xlsx")
df_fc = df_fc.rename(columns={
    "Material": COL_MATERIAL,
    "Nome Material": "Produto",
    "Custo Média Móvel": COL_PREV,
    "Mes": COL_MES,
})[[COL_CENTRO, COL_MATERIAL, COL_PREV, COL_MES]]

# ---------- Dedup dos DOIS lados (1 linha por chave) ----------
CHAVE = [COL_CENTRO, COL_MATERIAL, COL_MES]

for nome, df in [("BI", df_bi), ("Forecast", df_fc)]:
    n = df.duplicated(subset=CHAVE).sum()
    print(f"{nome}: {len(df):,} linhas | duplicatas por chave: {n:,}")

df_bi_d = df_bi.groupby(CHAVE, as_index=False).agg(
    {COL_REAL: "last", COL_PROD: "last", "Produto": "last"})
df_fc_d = df_fc.groupby(CHAVE, as_index=False).agg({COL_PREV: "last"})

# ---------- Chaves como texto nos TRÊS campos ----------
for c in CHAVE:
    df_bi_d[c] = df_bi_d[c].astype(str).str.strip()
    df_fc_d[c] = df_fc_d[c].astype(str).str.strip()

# ---------- Merge ----------
df_merged = df_bi_d.merge(df_fc_d, on=CHAVE, how="inner")

print(f"\nApós dedup — BI: {len(df_bi_d):,} | Forecast: {len(df_fc_d):,}")
print(f"Merge (inner): {len(df_merged):,} linhas")
print(f"Só no BI:       {len(df_bi_d) - len(df_merged):,}")
print(f"Só no forecast: {len(df_fc_d) - len(df_merged):,}")
display(df_merged.head())

## 3. Preparação

Converte o mês (nomes por extenso não são reconhecidos pelo `to_datetime`), calcula os erros, monta a curva ABC e aplica a tolerância.

**Sobre o baseline:** é o custo real do mês *imediatamente* anterior, calculado aqui com `shift` sobre a data. Não confundir com a coluna "Custo Média Móvel M-1" do arquivo de forecast — aquela é a média móvel anterior do próprio script, não o realizado.

In [ ]:
base = df_merged.copy()

# ---------- Numéricos ----------
for c in [COL_PREV, COL_REAL, COL_PROD]:
    base[c] = pd.to_numeric(base[c], errors="coerce")

# ---------- Mês por extenso -> data ordenável ----------
def _chave_mes(s):
    s = unicodedata.normalize("NFKD", str(s).strip().lower())
    return s.encode("ascii", "ignore").decode("ascii")

base["_mes_num"] = base[COL_MES].map(lambda x: MESES_NUM.get(_chave_mes(x)))
nao_conv = base["_mes_num"].isna().sum()
if nao_conv:
    print(f"ATENÇÃO — {nao_conv} meses não convertidos:")
    display(base.loc[base["_mes_num"].isna(), COL_MES].value_counts())

base["_dt"] = pd.to_datetime(
    dict(year=ANO, month=base["_mes_num"], day=1), errors="coerce")

_ordem = [m.capitalize() for m, _ in sorted(MESES_NUM.items(), key=lambda kv: kv[1])]
base["_mes_lbl"] = base["_mes_num"].map({v: k.capitalize() for k, v in MESES_NUM.items()})
base["_mes_lbl"] = pd.Categorical(
    base["_mes_lbl"], categories=[m for m in _ordem if m in set(base["_mes_lbl"])],
    ordered=True)

# ---------- Linhas inválidas para o cálculo percentual ----------
n0 = len(base)
base = base[base[COL_PREV].notna() & base[COL_REAL].notna()
            & (base[COL_REAL] != 0) & base[COL_PROD].notna() & (base[COL_PROD] > 0)]
n_excl = n0 - len(base)

# ---------- Erros do script ----------
base["Erro_Un_R$"]        = base[COL_PREV] - base[COL_REAL]
base["Erro_Un_Abs_R$"]    = base["Erro_Un_R$"].abs()
base["Erro_%"]            = base["Erro_Un_R$"] / base[COL_REAL]
base["APE"]               = base["Erro_%"].abs()
base["Valor_Real_R$"]     = base[COL_REAL] * base[COL_PROD]
base["Valor_Prev_R$"]     = base[COL_PREV] * base[COL_PROD]
base["Erro_Valor_Abs_R$"] = base["Erro_Un_Abs_R$"] * base[COL_PROD]

# ---------- Baseline: custo real do mês anterior ----------
base = base.sort_values([COL_CENTRO, COL_MATERIAL, "_dt"]).reset_index(drop=True)
g = base.groupby([COL_CENTRO, COL_MATERIAL])
base[COL_BASE] = g[COL_REAL].shift(1)
_ant = g["_dt"].shift(1)
base["_gap"] = ((base["_dt"].dt.year * 12 + base["_dt"].dt.month)
                - (_ant.dt.year * 12 + _ant.dt.month))

base["Erro_Un_Abs_M1"]    = (base[COL_BASE] - base[COL_REAL]).abs()
base["APE_M1"]            = base["Erro_Un_Abs_M1"] / base[COL_REAL]
base["Valor_M1_R$"]       = base[COL_BASE] * base[COL_PROD]
base["Erro_Valor_Abs_M1"] = base["Erro_Un_Abs_M1"] * base[COL_PROD]

# ---------- Curva ABC pelo valor movimentado ----------
valor_mat = (base.groupby([COL_CENTRO, COL_MATERIAL])["Valor_Real_R$"].sum()
             .sort_values(ascending=False).reset_index())
valor_mat["Acum"] = valor_mat["Valor_Real_R$"].cumsum() / valor_mat["Valor_Real_R$"].sum()
valor_mat["Classe_ABC"] = np.select(
    [valor_mat["Acum"] <= CORTE_A, valor_mat["Acum"] <= CORTE_B], ["A", "B"], "C")
base = base.merge(valor_mat[[COL_CENTRO, COL_MATERIAL, "Classe_ABC"]],
                  on=[COL_CENTRO, COL_MATERIAL], how="left")

# ---------- Tolerância ----------
def aplicar_tolerancia(df, ape_col="APE", erro_un_col="Erro_Un_Abs_R$",
                       modo=MODO_TOLERANCIA, regra=REGRA_COMBINADO):
    pct = df[ape_col] <= df["Classe_ABC"].map(TOLERANCIAS)
    absl = df[erro_un_col] <= df["Classe_ABC"].map(TOLERANCIAS_ABS)
    if modo == "percentual":
        return pct
    if modo == "absoluto":
        return absl
    if modo == "combinado":
        return (pct | absl) if regra == "ou" else (pct & absl)
    raise ValueError(f"modo inválido: {modo!r}")

base["Acertou"]    = aplicar_tolerancia(base)
base["Acertou_M1"] = aplicar_tolerancia(base, "APE_M1", "Erro_Un_Abs_M1")

print(f"Linhas analisadas: {len(base):,} (excluídas: {n_excl:,})")
print(f"Materiais (Centro+Código): "
      f"{base[[COL_CENTRO, COL_MATERIAL]].drop_duplicates().shape[0]:,}")
print(f"Modo de tolerância: {MODO_TOLERANCIA}")

## 4. Verificações de integridade

Rode sempre. Se algo aqui destoar, os indicadores das seções seguintes não são confiáveis.

In [ ]:
chk = []

chk.append(("Duplicatas por chave em base",
            base.duplicated(subset=[COL_CENTRO, COL_MATERIAL, COL_MES]).sum(), "= 0"))
chk.append(("Meses não convertidos", int(base["_mes_num"].isna().sum()), "= 0"))
chk.append(("Custo real <= 0", int((base[COL_REAL] <= 0).sum()), "= 0"))
chk.append(("Produção <= 0", int((base[COL_PROD] <= 0).sum()), "= 0"))
chk.append(("APE acima de 100%", int((base["APE"] > 1).sum()), "poucos"))
display(pd.DataFrame(chk, columns=["Verificação", "Resultado", "Esperado"])
        .style.hide(axis="index"))

print("\nEscala das colunas críticas (previsto e real devem ter ordem de grandeza igual):")
display(base[[COL_PREV, COL_REAL, COL_BASE, COL_PROD]].describe().round(2))

print("\nCobertura mensal por material:")
display(base.groupby([COL_CENTRO, COL_MATERIAL])[COL_MES].nunique()
        .value_counts().sort_index().to_frame("Materiais").rename_axis("Meses com dado"))

print("\nDistância até o mês anterior (baseline usa apenas gap = 1):")
display(base["_gap"].value_counts(dropna=False).to_frame("Linhas").rename_axis("Gap"))

## 5. Painel executivo

In [ ]:
wmape    = base["Erro_Valor_Abs_R$"].sum() / base["Valor_Real_R$"].sum()
acuracia = 1 - wmape
mae_un   = base["Erro_Un_Abs_R$"].mean()
vies     = (base["Valor_Prev_R$"].sum() - base["Valor_Real_R$"].sum()) / base["Valor_Real_R$"].sum()
hit_rate = base["Acertou"].mean()

painel = pd.DataFrame({
    "Indicador": ["Acurácia (1−WMAPE)", "MAE — erro no preço unitário", "Viés", "Hit Rate"],
    "Resultado": [f"{acuracia:.1%}", f"R$ {mae_un:,.2f} /un",
                  f"{vies:+.1%}", f"{hit_rate:.1%}"],
    "Leitura": [
        "Ponderado pelo valor movimentado (custo real × produção)",
        "Distância média entre preço previsto e real, por material/mês",
        "Positivo = superestima | Negativo = subestima",
        f"% dentro da tolerância (A ±{TOLERANCIAS['A']:.0%} | B ±{TOLERANCIAS['B']:.0%} | C ±{TOLERANCIAS['C']:.0%})",
    ],
})
display(painel.style.hide(axis="index"))

print(f"Acertos: {int(base['Acertou'].sum()):,} de {len(base):,} previsões")
print(f"Valor movimentado no período: R$ {base['Valor_Real_R$'].sum():,.0f}")

## 6. Acurácia por classe de material

A classe A concentra ~80% do dinheiro — é o número que define se o forecast é confiável na prática.

**Cuidado ao comparar Hit Rate entre classes:** a classe A é julgada com régua de ±5% e a C com ±20%, então um hit rate menor em A não significa forecast pior. A acurácia é a métrica comparável; o hit rate não é.

In [ ]:
por_classe = (
    base.groupby("Classe_ABC")
        .apply(lambda x: pd.Series({
            "Materiais":  x[COL_MATERIAL].nunique(),
            "Valor_Real": x["Valor_Real_R$"].sum(),
            "Acuracia":   1 - x["Erro_Valor_Abs_R$"].sum() / x["Valor_Real_R$"].sum(),
            "MAE_Un_R$":  x["Erro_Un_Abs_R$"].mean(),
            "Vies":       (x["Valor_Prev_R$"].sum() - x["Valor_Real_R$"].sum()) / x["Valor_Real_R$"].sum(),
            "Hit_Rate":   x["Acertou"].mean(),
        }), include_groups=False)
        .reset_index()
)
por_classe["Materiais"]  = por_classe["Materiais"].astype(int)
por_classe["%_do_Valor"] = por_classe["Valor_Real"] / por_classe["Valor_Real"].sum()

display(
    por_classe[["Classe_ABC", "Materiais", "%_do_Valor", "Acuracia",
                "MAE_Un_R$", "Vies", "Hit_Rate"]]
    .style.hide(axis="index")
    .format({"Materiais": "{:,.0f}", "%_do_Valor": "{:.1%}", "Acuracia": "{:.1%}",
             "MAE_Un_R$": "R$ {:,.2f}", "Vies": "{:+.1%}", "Hit_Rate": "{:.1%}"})
)

# Conferência: a média das classes ponderada pelo valor reconcilia com o painel
recon = 1 - (por_classe["%_do_Valor"] * (1 - por_classe["Acuracia"])).sum()
print(f"Reconciliação com o painel geral: {recon:.2%} (esperado {acuracia:.2%})")

## 7. O script agrega valor? — comparação com baseline

Um número de acurácia isolado não diz se o script vale o esforço. O baseline é a alternativa mais simples possível: **assumir que o custo do mês é igual ao do mês anterior**.

Os dois competidores são medidos contra o mesmo custo real, com a mesma fórmula — só muda quem faz o papel de "previsto". A comparação usa apenas linhas com gap = 1 (mês imediatamente anterior).

In [ ]:
comp = base[base[COL_BASE].notna() & (base["_gap"] == 1)].copy()

acur_s = 1 - comp["Erro_Valor_Abs_R$"].sum() / comp["Valor_Real_R$"].sum()
acur_b = 1 - comp["Erro_Valor_Abs_M1"].sum() / comp["Valor_Real_R$"].sum()
hit_s, hit_b = comp["Acertou"].mean(), comp["Acertou_M1"].mean()
mae_s, mae_b = comp["Erro_Un_Abs_R$"].mean(), comp["Erro_Un_Abs_M1"].mean()
err_s, err_b = comp["Erro_Valor_Abs_R$"].sum(), comp["Erro_Valor_Abs_M1"].sum()

print(f"Linhas comparáveis: {len(comp):,} de {len(base):,} ({len(comp)/len(base):.0%})")
print(f"Cobrem {comp['Valor_Real_R$'].sum() / base['Valor_Real_R$'].sum():.0%} do valor movimentado")
print(f"Sem baseline aplicável: {len(base) - len(comp):,} linhas "
      f"({1 - len(comp)/len(base):.0%}) — material novo ou produção intermitente\n")

display(pd.DataFrame({
    "Indicador": ["Acurácia", "Hit Rate", "MAE (R$/un)", "Erro total"],
    "Baseline (mês anterior)": [f"{acur_b:.1%}", f"{hit_b:.1%}",
                                f"R$ {mae_b:,.2f}", f"R$ {err_b:,.0f}"],
    "Script": [f"{acur_s:.1%}", f"{hit_s:.1%}",
               f"R$ {mae_s:,.2f}", f"R$ {err_s:,.0f}"],
    "Diferença": [f"{(acur_s - acur_b)*100:+.2f} p.p.", f"{(hit_s - hit_b)*100:+.1f} p.p.",
                  f"R$ {mae_s - mae_b:+,.2f}", f"R$ {err_s - err_b:+,.0f}"],
}).style.hide(axis="index"))

comp["Script_venceu"] = comp["Erro_Valor_Abs_R$"] < comp["Erro_Valor_Abs_M1"]
print(f"\nScript melhor em {int(comp['Script_venceu'].sum())} linhas "
      f"({comp['Script_venceu'].mean():.1%}) | "
      f"Baseline melhor em {int((~comp['Script_venceu']).sum())} "
      f"({1 - comp['Script_venceu'].mean():.1%})")
print(f"Desvio evitado no período: R$ {err_b - err_s:,.0f}")

### Onde o script ganha e o quanto o ganho é concentrado

In [ ]:
por_classe_comp = (
    comp.groupby("Classe_ABC")
        .apply(lambda x: pd.Series({
            "Linhas":         len(x),
            "Script_venceu":  x["Script_venceu"].mean(),
            "Acur_Baseline":  1 - x["Erro_Valor_Abs_M1"].sum() / x["Valor_Real_R$"].sum(),
            "Acur_Script":    1 - x["Erro_Valor_Abs_R$"].sum() / x["Valor_Real_R$"].sum(),
            "R$_evitado":     x["Erro_Valor_Abs_M1"].sum() - x["Erro_Valor_Abs_R$"].sum(),
        }), include_groups=False)
        .reset_index()
)
por_classe_comp["Ganho_pp"] = (por_classe_comp["Acur_Script"]
                               - por_classe_comp["Acur_Baseline"]) * 100
display(por_classe_comp.style.hide(axis="index")
        .format({"Linhas": "{:,.0f}", "Script_venceu": "{:.1%}",
                 "Acur_Baseline": "{:.1%}", "Acur_Script": "{:.1%}",
                 "R$_evitado": "R$ {:,.0f}", "Ganho_pp": "{:+.2f}"}))

# Concentração do ganho: poucos materiais explicam tudo, ou é distribuído?
ganho = (comp.groupby([COL_CENTRO, COL_MATERIAL, "Classe_ABC"])
             .apply(lambda x: pd.Series({
                 "Ganho_R$": x["Erro_Valor_Abs_M1"].sum() - x["Erro_Valor_Abs_R$"].sum()
             }), include_groups=False)
             .reset_index().sort_values("Ganho_R$", ascending=False))
ganho["%_acum"] = ganho["Ganho_R$"].cumsum() / ganho["Ganho_R$"].sum()

n50 = int((ganho["%_acum"] <= 0.50).sum()) + 1
print(f"\n{n50} material(is) de {len(ganho)} concentram 50% do ganho "
      f"({n50/len(ganho):.0%} dos materiais)")
display(ganho.head(10).style.hide(axis="index")
        .format({"Ganho_R$": "R$ {:,.0f}", "%_acum": "{:.1%}"}))

## 8. Onde está o dinheiro do erro

Materiais ordenados pelo impacto em R$ (erro no preço × volume produzido) — a lista de prioridade para investigação.

In [ ]:
top_ofensores = (
    base.groupby([COL_CENTRO, COL_MATERIAL, "Classe_ABC"])
        .agg(Valor_Real=("Valor_Real_R$", "sum"),
             Erro_Impacto=("Erro_Valor_Abs_R$", "sum"),
             Preco_Real_Medio=(COL_REAL, "mean"),
             Preco_Prev_Medio=(COL_PREV, "mean"),
             APE_Medio=("APE", "mean"))
        .reset_index().sort_values("Erro_Impacto", ascending=False)
)
top_ofensores["%_do_Erro"] = top_ofensores["Erro_Impacto"] / top_ofensores["Erro_Impacto"].sum()
top_ofensores["%_Acum"]    = top_ofensores["%_do_Erro"].cumsum()

display(top_ofensores.head(TOP_N).style.hide(axis="index")
        .format({"Valor_Real": "R$ {:,.0f}", "Erro_Impacto": "R$ {:,.0f}",
                 "Preco_Real_Medio": "R$ {:,.4f}", "Preco_Prev_Medio": "R$ {:,.4f}",
                 "APE_Medio": "{:.1%}", "%_do_Erro": "{:.1%}", "%_Acum": "{:.1%}"}))

### Diagnóstico detalhado por classe

A coluna `Meses_Erro` mostra **quando** cada material estourou a tolerância. O padrão dá o diagnóstico:

- Meses consecutivos no início → parâmetro que foi corrigido depois, ou efeito de estoque inicial
- Meses consecutivos no fim → algo mudou no meio do caminho (reajuste, troca de BOM)
- Todos os meses → cadastro ou custo de componente desatualizado (mais fácil de corrigir)
- Meses esparsos → flutuação de mercado, provavelmente fora do controle do script

In [ ]:
def diagnostico_classe(df, classe, top_n=10):
    sub = df[df["Classe_ABC"] == classe].sort_values("_dt").copy()
    sub["_mes_erro"] = np.where(~sub["Acertou"], sub["_mes_lbl"].astype(str).str[:3], "")

    g = (sub.groupby([COL_CENTRO, COL_MATERIAL])
            .agg(Meses=(COL_MES, "nunique"),
                 Acertos=("Acertou", "sum"),
                 Valor_Real=("Valor_Real_R$", "sum"),
                 Valor_Prev=("Valor_Prev_R$", "sum"),
                 Erro_Impacto=("Erro_Valor_Abs_R$", "sum"),
                 Meses_Erro=("_mes_erro", lambda s: ", ".join(m for m in s if m)),
                 Pior_APE=("APE", "max"))
            .reset_index().sort_values("Erro_Impacto", ascending=False))

    g["Acuracia"]    = 1 - g["Erro_Impacto"] / g["Valor_Real"]
    g["Vies"]        = (g["Valor_Prev"] - g["Valor_Real"]) / g["Valor_Real"]
    g["%_Erro_Acum"] = g["Erro_Impacto"].cumsum() / g["Erro_Impacto"].sum()
    g["Acertou_em"]  = g["Acertos"].astype(int).astype(str) + "/" + g["Meses"].astype(str)

    n50 = int((g["%_Erro_Acum"] <= 0.50).sum()) + 1
    print(f"Classe {classe}: {len(g)} materiais | "
          f"erro total R$ {g['Erro_Impacto'].sum():,.0f}")
    print(f"{n50} material(is) concentram 50% do erro da classe "
          f"({n50/len(g):.0%} dos materiais)")

    cols = [COL_CENTRO, COL_MATERIAL, "Valor_Real", "Erro_Impacto", "%_Erro_Acum",
            "Acuracia", "Vies", "Acertou_em", "Meses_Erro", "Pior_APE"]
    return (g.head(top_n)[cols].style.hide(axis="index")
             .format({"Valor_Real": "R$ {:,.0f}", "Erro_Impacto": "R$ {:,.0f}",
                      "%_Erro_Acum": "{:.1%}", "Acuracia": "{:.1%}",
                      "Vies": "{:+.1%}", "Pior_APE": "{:.1%}"}))

display(diagnostico_classe(base, "A"))

## 9. Visuais para a apresentação

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

cores = {"A": "#2a9d8f", "B": "#e9c46a", "C": "#e76f51"}
axes[0].bar(por_classe["Classe_ABC"], por_classe["Acuracia"],
            color=[cores[c] for c in por_classe["Classe_ABC"]])
axes[0].axhline(acuracia, color="gray", ls="--", lw=1, label=f"Geral: {acuracia:.1%}")
axes[0].set_title("Acurácia por classe de material")
axes[0].set_ylim(0, 1.05); axes[0].legend()
for i, v in enumerate(por_classe["Acuracia"]):
    axes[0].text(i, v + 0.02, f"{v:.1%}", ha="center", fontweight="bold")

vr, vp = base["Valor_Real_R$"].sum(), base["Valor_Prev_R$"].sum()
axes[1].bar(["Real", "Previsto"], [vr, vp], color=["#457b9d", "#a8dadc"])
axes[1].set_title(f"Valor movimentado (viés {vies:+.1%})")
for i, v in enumerate([vr, vp]):
    axes[1].text(i, v, f"R$ {v/1e6:,.1f}M", ha="center", va="bottom")

axes[2].bar(["Baseline\n(mês anterior)", "Script"], [err_b, err_s],
            color=["#adb5bd", "#2a9d8f"])
axes[2].set_title(f"Erro total — script evita R$ {(err_b-err_s)/1e6:,.1f}M")
for i, v in enumerate([err_b, err_s]):
    axes[2].text(i, v, f"R$ {v/1e6:,.1f}M", ha="center", va="bottom")

plt.tight_layout(); plt.show()

## 10. Exportação

In [ ]:
with pd.ExcelWriter("validacao_forecast_resultados.xlsx") as w:
    painel.to_excel(w, sheet_name="Painel", index=False)
    por_classe.to_excel(w, sheet_name="Por_Classe", index=False)
    por_classe_comp.to_excel(w, sheet_name="Baseline_Por_Classe", index=False)
    ganho.to_excel(w, sheet_name="Ganho_Por_Material", index=False)
    top_ofensores.to_excel(w, sheet_name="Top_Ofensores", index=False)
    base.drop(columns=["_mes_num", "_dt", "_mes_lbl"]).to_excel(
        w, sheet_name="Base_Detalhada", index=False)

print("Arquivo 'validacao_forecast_resultados.xlsx' gerado.")